# Model Training — Bank Account Fraud (BAF)

This notebook trains and compares four fraud detection models on the BAF NeurIPS 2022 dataset.

**Prerequisites:** Run `python DATA/preprocess.py` before this notebook to generate the processed arrays in `outputs/processed/`.

## 1. Imports & Config

In [ ]:
import os
import sys

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

PROCESSED_DIR = "../outputs/processed"
MODEL_OUT_DIR = "../outputs"
RANDOM_STATE = 42

os.makedirs(MODEL_OUT_DIR, exist_ok=True)

## 2. Load Processed Data

In [ ]:
if not os.path.exists(PROCESSED_DIR):
    sys.exit("outputs/processed/ not found. Run: python DATA/preprocess.py")

X_train = np.load(f"{PROCESSED_DIR}/X_train.npy", allow_pickle=True)
X_val   = np.load(f"{PROCESSED_DIR}/X_val.npy",   allow_pickle=True)
X_test  = np.load(f"{PROCESSED_DIR}/X_test.npy",  allow_pickle=True)
y_train = np.load(f"{PROCESSED_DIR}/y_train.npy", allow_pickle=True)
y_val   = np.load(f"{PROCESSED_DIR}/y_val.npy",   allow_pickle=True)
y_test  = np.load(f"{PROCESSED_DIR}/y_test.npy",  allow_pickle=True)

neg, pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg / pos

print(f"Train : {len(y_train):>8,}  fraud rate: {y_train.mean():.4f}")
print(f"Val   : {len(y_val):>8,}  fraud rate: {y_val.mean():.4f}")
print(f"Test  : {len(y_test):>8,}  fraud rate: {y_test.mean():.4f}")
print(f"\nscale_pos_weight: {scale_pos_weight:.1f}")

## 3. Define Models

Adjust hyperparameters here before running cross-validation.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "LightGBM": lgb.LGBMClassifier(
        scale_pos_weight=scale_pos_weight,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=-1,
    ),
    "XGBoost": xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        eval_metric="aucpr",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0,
    ),
}

## 4. Cross-Validation

5-fold stratified CV on the training set, scored by AUPRC.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print(f"{'Model':<22} {'CV AUPRC (mean±std)'}")
print("-" * 45)

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="average_precision", n_jobs=-1
    )
    cv_results[name] = scores
    print(f"{name:<22} {scores.mean():.4f} ± {scores.std():.4f}")

## 5. Train on Full Training Set & Score on Validation

In [ ]:
results = {}

print(f"{'Model':<22} {'CV AUPRC (mean±std)':<26} {'Val AUPRC'}")
print("-" * 65)

for name, model in models.items():
    model.fit(X_train, y_train)
    val_proba = model.predict_proba(X_val)[:, 1]
    val_auprc = average_precision_score(y_val, val_proba)
    scores = cv_results[name]
    results[name] = {"model": model, "val_auprc": val_auprc}
    print(f"{name:<22} {scores.mean():.4f} ± {scores.std():.4f}        {val_auprc:.4f}")

best_name = max(results, key=lambda k: results[k]["val_auprc"])
print(f"\nBest model: {best_name}  (Val AUPRC: {results[best_name]['val_auprc']:.4f})")

## 6. Validation Precision-Recall Curves

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(8, 6))
baseline = y_val.mean()
ax.axhline(baseline, color="gray", linestyle="--", label=f"Baseline (prevalence={baseline:.4f})")

for name, r in results.items():
    y_proba = r["model"].predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, y_proba)
    ap = average_precision_score(y_val, y_proba)
    ax.plot(recall, precision, label=f"{name} (AUPRC={ap:.3f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Validation Set")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 7. Save All Models

In [ ]:
for name, r in results.items():
    path = os.path.join(MODEL_OUT_DIR, f"{name}.joblib")
    joblib.dump(r["model"], path)
    print(f"Saved {name} → {path}")

print(f"\nBest model: {best_name}")
print("Run MODELS/evaluate.ipynb (or evaluate.py) for final test-set evaluation.")